In [ ]:
# Cell 1: Import required modules
import rospy
import actionlib
from ipywidgets import Button, FloatText, VBox, HBox, Label, Output
import ipywidgets as widgets 
from IPython.display import display, HTML
from assignment_2_2024.msg import PlanningAction, PlanningGoal, RobotState
from nav_msgs.msg import Odometry
from geometry_msgs.msg import PoseStamped
from sensor_msgs.msg import LaserScan
import threading
from threading import Timer
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
%matplotlib widget

In [ ]:
# Cell 2: Define the RobotActionClient class
class RobotActionClient:
    def __init__(self):
        self.ac = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
        self.ac.wait_for_server()
        
        self.pub = rospy.Publisher('/robot_state', RobotState, queue_size=10)
        self.odom_sub = rospy.Subscriber('/odom', Odometry, self.odom_callback)
        self.robot_state = RobotState()
        self.output = Output()

    def send_goal(self, x, y):
        goal = PlanningGoal()
        goal.target_pose = PoseStamped()
        goal.target_pose.header.frame_id = "map"
        goal.target_pose.header.stamp = rospy.Time.now()
        goal.target_pose.pose.position.x = x
        goal.target_pose.pose.position.y = y
        goal.target_pose.pose.orientation.w = 1.0
        
        self.ac.send_goal(goal, 
                         done_cb=self.done_callback,
                         active_cb=self.active_callback,
                         feedback_cb=self.feedback_callback)
        with self.output:
            print(f"Goal sent to position (x: {x:.2f}, y: {y:.2f})")

    def cancel_goal(self):
        self.ac.cancel_goal()
        with self.output:
            print("Goal canceled.")

    def done_callback(self, status, result):
        with self.output:
            print(f"Goal completed with status: {status}")

    def active_callback(self):
        with self.output:
            print("Goal is now active.")

    def feedback_callback(self, feedback):
        pass

    def odom_callback(self, msg):
        self.robot_state.x = msg.pose.pose.position.x
        self.robot_state.y = msg.pose.pose.position.y
        self.robot_state.vel_x = msg.twist.twist.linear.x
        self.robot_state.vel_z = msg.twist.twist.angular.z
        self.pub.publish(self.robot_state)

In [ ]:
# Cell 3: Initialize ROS node and client
rospy.init_node('robot_action_client_jupyter', anonymous=True)
client = RobotActionClient()

# Start spinner thread
spinner = threading.Thread(target=rospy.spin)
spinner.daemon = True
spinner.start()

In [ ]:
# Cell 4: Create widgets
x_input = FloatText(description='X:')
y_input = FloatText(description='Y:')
send_btn = Button(description='Send Goal')
cancel_btn = Button(description='Cancel Goal')
exit_btn = Button(description='Exit')
output = client.output

# Create layout
controls = VBox([
    HBox([x_input, y_input]),
    HBox([send_btn, cancel_btn, exit_btn]),
    output
])

In [ ]:
# Cell 5: Define button click handlers
def on_send_click(b):
    try:
        x = x_input.value
        y = y_input.value
        client.send_goal(x, y)
    except ValueError:
        with output:
            print("Invalid input! Please enter numeric values.")

def on_cancel_click(b):
    client.cancel_goal()

def on_exit_click(b):
    rospy.signal_shutdown("User requested exit.")
    with output:
        print("Shutting down...")
    exit_btn.disabled = True

send_btn.on_click(on_send_click)
cancel_btn.on_click(on_cancel_click)
exit_btn.on_click(on_exit_click)

In [ ]:
# Cell 7: Add LaserScan subscriber and distance monitoring
class LaserSubscriber:
    def __init__(self):
        self.closest_obstacle = float('inf')
        self.laser_sub = rospy.Subscriber('/scan', LaserScan, self.laser_callback)
        
    def laser_callback(self, msg):
        # Calculate minimum distance from all regions
        if msg.ranges:
            self.closest_obstacle = min(min(msg.ranges), 10)  # Cap at 10 meters

# Add to existing Cell 3 after client initialization:
laser_listener = LaserSubscriber()

In [ ]:
# Create styled status widgets
status_title = widgets.HTML(
    value='<h3 style="color:#2c3e50; margin-bottom:0;">Robot Status Monitor</h3>'
)

position_display = widgets.HTML(
    value='<span style="color:#27ae60;">Position: </span>(0.00, 0.00) m',
    layout=widgets.Layout(margin='0 20px')
)

obstacle_display = widgets.HTML(
    value='<span style="color:#e67e22;">Closest Obstacle: </span>∞ m',
    layout=widgets.Layout(margin='0 20px')
)

# Create grid container
status_grid = widgets.GridBox(
    children=[position_display, obstacle_display],
    layout=widgets.Layout(
        grid_template_columns='repeat(2, 300px)',
        border='2px solid #bdc3c7',
        padding='15px',
        margin='20px 0',
        border_radius='8px',
        background='#f8f9fa'
    )
)

# Combine all status elements
status_panel = widgets.VBox(
    [status_title, status_grid],
    layout=widgets.Layout(margin='30px 0 0 0')
)

#display(status_panel)

# Update function
def update_display():
    try:
        # Get current values
        x = client.robot_state.x
        y = client.robot_state.y
        distance = laser_listener.closest_obstacle
        
        # Update position
        position_display.value = f'''
            <span style="color:#27ae60; font-weight:bold;">Position: </span>
            <span style="font-family: monospace;">({x:.3f}, {y:.3f}) m</span>
        '''
        
        # Update obstacle distance with warning
        obstacle_text = f'{distance:.2f} m'
        if distance < 1.0:
            obstacle_text = f'<span style="color:#e74c3c;">{obstacle_text} ⚠</span>'
            
        obstacle_display.value = f'''
            <span style="color:#e67e22; font-weight:bold;">Closest Obstacle: </span>
            <span style="font-family: monospace;">{obstacle_text}</span>
        '''
    
    except Exception as e:
        pass
    
    # Restart timer
    Timer(0.1, update_display).start()

In [ ]:
# Cell 8: plotting preparation

# Initialize plot data
class PlotData:
    def __init__(self):
        self.x_pos = []
        self.y_pos = []
        self.reached_goals = 0
        self.not_reached_goals = 0

plot_data = PlotData()

# Modify the done callback in RobotActionClient class (Cell 2)
def done_callback(self, status, result):
    with self.output:
        print(f"Goal completed with status: {status}")
    if status == 3:  # SUCCEEDED
        plot_data.reached_goals += 1
    else:  # PREEMPTED(2) or other statuses
        plot_data.not_reached_goals += 1

# Update the class definition
RobotActionClient.done_callback = done_callback

In [ ]:
# Cell 9: Create and configure the plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Robot Monitoring Dashboard', fontsize=14)

# Trajectory plot
ax1.set_title('Robot Trajectory')
ax1.set_xlabel('X Position')
ax1.set_ylabel('Y Position')
traj_line, = ax1.plot([], [], 'b-', lw=1)
current_pos, = ax1.plot([], [], 'ro', markersize=8)
ax1.grid(True)

# Goals status plot
ax2.set_title('Goals Achievement')
ax2.set_ylabel('Count')
goal_bars = ax2.bar([0, 1], [0, 0], tick_label=['Reached', 'Not Reached'], color=['green', 'red'])

# Set consistent limits
ax1.set_xlim(-10, 10)
ax1.set_ylim(-10, 10)
ax2.set_ylim(0, 10)

def update_plots(frame):
    # Update trajectory
    if client.robot_state.x and client.robot_state.y:
        plot_data.x_pos.append(client.robot_state.x)
        plot_data.y_pos.append(client.robot_state.y)
        traj_line.set_data(plot_data.x_pos, plot_data.y_pos)
        current_pos.set_data([client.robot_state.x], [client.robot_state.y])
    
    # Update goal bars
    for bar, height in zip(goal_bars, [plot_data.reached_goals, plot_data.not_reached_goals]):
        bar.set_height(height)
    ax2.set_ylim(0, max(1, plot_data.reached_goals + plot_data.not_reached_goals + 1))
    
    return traj_line, current_pos, *goal_bars

# Create animation
ani = FuncAnimation(fig, update_plots, interval=500, blit=True, cache_frame_data=False)
# Cell 10: Display all components together
update_display()
display(status_panel)
display(controls)
plt.show()